In [ ]:
import pandas as pd
import numpy as np
import json
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import joblib
import warnings

warnings.filterwarnings('ignore')

print("🚀 启动 [DNN vs SVR] 双模型轨迹预测与终点容量对比对决矩阵...\n")

# ==========================================
# 1. 终极数据加载与时空大展开
# ==========================================
# 读取由第一个 Notebook 洗出来的终极特征 CSV
df = pd.read_csv("LFP_Ultimate_Features_and_Trajectory.csv", index_col=0)

# 定义 16 维静态特征：4维工况 + 12维物理基因
feature_cols = [
    'Cond_Temp', 'Cond_DOD', 'Cond_SOC', 'Cond_C_Rate',
    'F1_Charge_Time_Diff', 'F8_Discharge_Time_Diff', 'F7_IC_Area_Diff', 
    'F9_Var_Time_Diff', 'F17_Mean_Cap_Diff', 'F22_Linear_Intercept', 
    'F24_Sqrt_Intercept', 'F34_Temp_Time_Integral_Diff', 'F35_Min_Temp_Diff', 
    'F38_Avg_Min_Temp', 'F39_Avg_Mean_Temp', 'F41_DCIR_Diff'
]

expanded_data = []
print("📦 正在解锁 80 块电池的生命档案，执行时空打点展开...")

for index, row in df.iterrows():
    trajectory = json.loads(row['Y_Trajectory'])
    base_features = row[feature_cols].values.astype(float)
    
    # 将 JSON 里的每一对 [圈数: 容量] 拆解为一行独立的纵向训练样本
    for cycle_k_str, cap_value in trajectory.items():
        cycle_k = float(cycle_k_str)
        # 拼接为 17 维输入特征向量: [16维固定基因, 查询圈数 k]
        sample_x = np.append(base_features, cycle_k)
        expanded_data.append((sample_x, cap_value, index))

columns_x = feature_cols + ['Query_Cycle_k']
expanded_df = pd.DataFrame([item[0] for item in expanded_data], columns=columns_x)
y_target = np.array([item[1] for item in expanded_data])
battery_ids = np.array([item[2] for item in expanded_data])

print(f"✅ 降维打击成功！全量数据已展开为 {len(expanded_df)} 个特征打点样本。")

# ==========================================
# 2. 统一尺度标准化
# ==========================================
X_raw = expanded_df.values
y_raw = y_target

scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X_raw)

# 物理封存标尺
joblib.dump(scaler_X, 'lfp_battle_scaler.pkl')

# ==========================================
# 3. 战场 A：SVR (支持向量回归) 极速训练
# ==========================================
print("\n🤖 正在激活支持向量机，使用 RBF 高斯核函数进行高维超平面拟合...")
# 使用径向基核函数(RBF)，惩罚系数 C 和 Gamma 设为经典的工业稳健参数
svr_model = SVR(kernel='rbf', C=20.0, epsilon=0.01, gamma='scale')
svr_model.fit(X_scaled, y_raw)
print("✅ SVR 基线模型全量拟合完毕！")

# ==========================================
# 4. 战场 B：DNN (条件深度网络) 强力收敛
# ==========================================
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_raw, dtype=torch.float32).view(-1, 1)

class ConditionalTrajectoryNet(nn.Module):
    def __init__(self, num_static_features):
        super(ConditionalTrajectoryNet, self).__init__()
        # 物理编码器
        self.physics_encoder = nn.Sequential(
            nn.Linear(num_static_features, 64),
            nn.LeakyReLU(0.01),
            nn.Linear(64, 32),
            nn.LeakyReLU(0.01)
        )
        # 轨迹解码器
        self.trajectory_decoder = nn.Sequential(
            nn.Linear(33, 64),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.05),
            nn.Linear(64, 32),
            nn.LeakyReLU(0.01),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        static_features = x[:, :-1]
        query_cycle_k = x[:, -1:]
        hidden_physics = self.physics_encoder(static_features)
        combined = torch.cat((hidden_physics, query_cycle_k), dim=1)
        return self.trajectory_decoder(combined)

dnn_model = ConditionalTrajectoryNet(num_static_features=len(feature_cols))
criterion = nn.MSELoss()
optimizer = optim.Adam(dnn_model.parameters(), lr=0.001, weight_decay=1e-5)

epochs = 1200
print(f"\n⚙️ 正在启动 PyTorch 深度条件网络，执行 {epochs} 轮次迭代训练...")

for epoch in range(epochs):
    dnn_model.train()
    optimizer.zero_grad()
    predictions = dnn_model(X_tensor)
    loss = criterion(predictions, y_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 300 == 0:
        print(f"   -> Epoch [{epoch+1}/{epochs}], DNN 实时拟合 MSE 损失: {loss.item():.6f}")

torch.save(dnn_model.state_dict(), 'lfp_battle_dnn_weights.pth')
print("✅ DNN 深度模型权重全量封存完毕！")

# ==========================================
# 5. 见证奇迹：真实提取 vs DNN预测曲线 vs SVR预测曲线
# ==========================================
dnn_model.eval()

# 挑选两块典型的电池来进行“全景大对决”绘图审阅（可自由更换ID）
test_battery_list = list(df.index[:2])

plt.figure(figsize=(14, 7))
# 使用高雅的莫兰迪/工业质感配色
colors = ['#1f77b4', '#d62728'] 

print("\n📊 正在隔空生成全生命周期预测曲线与终点容量考核指标...")

comparison_reports = []

with torch.no_grad():
    for idx, b_id in enumerate(test_battery_list):
        batt_row = df.loc[b_id]
        true_traj = json.loads(batt_row['Y_Trajectory'])
        
        # 抓取真实的时间与容量序列
        true_cycles = np.array([float(k) for k in true_traj.keys()])
        true_caps = np.array(list(true_traj.values()))
        
        # 物理真理终点
        true_final_cap = batt_row['Y_Final_Capacity']
        max_actual_cycle = max(true_cycles)
        
        # 密集生成连续的 X 轴圈数（从 100 圈一直外推至生命结束甚至延申）
        dense_cycles = np.linspace(100, max_actual_cycle, 100)
        
        # 提取固定特征
        base_feat = batt_row[feature_cols].values.astype(float)
        
        dnn_pred_curve = []
        svr_pred_curve = []
        
        # 循环让两个模型在每一个密集的圈数上执行盲测
        for k in dense_cycles:
            q_vector = np.append(base_feat, k).reshape(1, -1)
            q_scaled = scaler_X.transform(q_vector)
            
            # SVR 盲测输出
            svr_cap = svr_model.predict(q_scaled)[0]
            svr_pred_curve.append(svr_cap)
            
            # DNN 盲测输出
            q_tensor = torch.tensor(q_scaled, dtype=torch.float32)
            dnn_cap = dnn_model(q_tensor).item()
            dnn_pred_curve.append(dnn_cap)
            
        # 评估寿命终点（最大圈数那一刻）的容量预测
        final_q_vector = np.append(base_feat, max_actual_cycle).reshape(1, -1)
        final_q_scaled = scaler_X.transform(final_q_vector)
        
        final_pred_svr = svr_model.predict(final_q_scaled)[0]
        final_pred_dnn = dnn_model(torch.tensor(final_q_scaled, dtype=torch.float32)).item()
        
        # 记录报表
        comparison_reports.append({
            'Battery_ID': b_id,
            'True_Final_Cap(Ah)': round(true_final_cap, 4),
            'DNN_Final_Cap(Ah)': round(final_pred_dnn, 4),
            'DNN_Error(%)': round(abs(final_pred_dnn - true_final_cap)/true_final_cap * 100, 2),
            'SVR_Final_Cap(Ah)': round(final_pred_svr, 4),
            'SVR_Error(%)': round(abs(final_pred_svr - true_final_cap)/true_final_cap * 100, 2)
        })
        
        # --- 绘图流 ---
        # 真实提取打点 (散点)
        plt.scatter(true_cycles, true_caps, color=colors[idx], alpha=0.5, s=60, edgecolors='k', label=f'True Ground-Truth (ID: {b_id})')
        # DNN 连续曲线 (实线)
        plt.plot(dense_cycles, dnn_pred_curve, color=colors[idx], linestyle='-', linewidth=3, label=f'Conditional DNN Curve (ID: {b_id})')
        # SVR 连续曲线 (虚线)
        plt.plot(dense_cycles, svr_pred_curve, color=colors[idx], linestyle='--', linewidth=2, alpha=0.8, label=f'RBF-SVR Baseline Curve (ID: {b_id})')

# 美化画布
plt.title("🏆 Full-Lifespan Capacity Degradation Profile: DNN vs SVR vs Ground-Truth", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Cycle Number (k)", fontsize=12, labelpad=10)
plt.ylabel("Discharge Capacity (Ah)", fontsize=12, labelpad=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='best', frameon=True, facecolor='white', edgecolor='gainsboro')
plt.tight_layout()
plt.show()

# ==========================================
# 6. 输出定量考核报表 (最终容量硬核对比)
# ==========================================
print("\n" + "="*85)
print("📋 寿命终点（Final Cycle）容量预测精度硬核对决报表")
print("="*85)
report_df = pd.DataFrame(comparison_reports)
print(report_df.to_string(index=False))
print("="*85)
print("💡 结论提示：通过对比实线(DNN)与虚线(SVR)对散点(真实值)的逼近程度，以及误差报表中的 Error 比例，你可以完美量化你提取的16维特征在不同架构下的表达威力！")